# Retail Store Sales — Exploración y limpieza reproducible

**Objetivo:** preparar un dataset de ventas con valores faltantes para analizar ingresos, productos, clientes, canales y evolución temporal en SQL Server y Power BI.

Este notebook **reconstruye y consolida las transformaciones efectuadas durante el proyecto**, sin repetir las pruebas exploratorias ni los intentos de código que se corrigieron. No inventa cantidades, importes ni descuentos. El archivo original se conserva intacto.

**Datos de referencia:** 12.575 filas, 11 columnas originales, fechas entre 2022-01-01 y 2025-01-18. El CSV original debe copiarse manualmente en `data/raw/retail_store_sales.csv`; no se incluye por defecto en el repositorio.

> Importante: algunos valores de `Item` se infieren de una asociación unívoca de categoría y precio **dentro de los datos observados**. No equivale a validarlos contra un catálogo maestro.

## 1. Cargar librerías y localizar el archivo

Se utiliza Pandas para inspeccionar y transformar registros, y NumPy únicamente para verificar cálculos numéricos con tolerancia.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

# Funciona si Jupyter se abre desde la raíz del proyecto o desde notebooks/.
cwd = Path.cwd()
if (cwd / "notebooks").is_dir():
    raiz = cwd
elif cwd.name == "notebooks" and (cwd.parent / "README.md").exists():
    raiz = cwd.parent
else:
    raise FileNotFoundError("Abrí Jupyter desde retail-sales-analysis/ o notebooks/.")

origen = raiz / "data" / "raw" / "retail_store_sales.csv"
destino = raiz / "data" / "processed" / "retail_store_sales_clean.csv"
if not origen.is_file():
    raise FileNotFoundError(f"No se encontró el CSV original: {origen}")

df = pd.read_csv(origen)
print(f"Archivo de origen: {origen.name} | Filas: {len(df):,} | Columnas: {df.shape[1]}")
display(df.head(5))

Archivo de origen: retail_store_sales.csv | Filas: 12,575 | Columnas: 11


,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,NaN
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False


## 2. Perfilado inicial y patrones de nulos

Antes de modificar los datos se inspeccionan tipos, cantidades de nulos y duplicados, así como la coincidencia de faltantes en `Item`, `Price Per Unit`, `Quantity` y `Total Spent`. Esto permite decidir qué registros pueden recuperarse sin inventar información.

In [2]:
columnas_requeridas = [
    "Transaction ID", "Customer ID", "Category", "Item", "Price Per Unit",
    "Quantity", "Total Spent", "Payment Method", "Location",
    "Transaction Date", "Discount Applied",
]
faltantes = set(columnas_requeridas) - set(df.columns)
if faltantes:
    raise ValueError(f"El CSV no contiene las columnas esperadas: {sorted(faltantes)}")

perfil = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "nulos": df.isna().sum(),
    "porcentaje_nulos": (df.isna().mean() * 100).round(2),
    "valores_unicos": df.nunique(dropna=True),
})
display(perfil)

print("Filas duplicadas:", int(df.duplicated().sum()))
print("Transaction ID repetidos:", int(df["Transaction ID"].duplicated().sum()))
print("\nPatrones conjuntos de campos faltantes (True = falta el valor):")
display(df[["Item", "Price Per Unit", "Quantity", "Total Spent"]].isna().value_counts())

,tipo,nulos,porcentaje_nulos,valores_unicos
Transaction ID,object,0,0.00,12575
Customer ID,object,0,0.00,25
Category,object,0,0.00,8
Item,object,1213,9.65,200
Price Per Unit,float64,609,4.84,25
Quantity,float64,604,4.80,10
Total Spent,float64,604,4.80,227
Payment Method,object,0,0.00,3
Location,object,0,0.00,2
Transaction Date,object,0,0.00,1114


Filas duplicadas: 0
Transaction ID repetidos: 0

Patrones conjuntos de campos faltantes (True = falta el valor):


Item   Price Per Unit  Quantity  Total Spent
False  False           False     False          11362
True   True            False     False            609
       False           True      True             604
Name: count, dtype: int64

## 3. Recuperación controlada de productos

**Regla aplicada:** completar `Item` únicamente cuando la combinación `(Category, Price Per Unit)` aparece asociada a **un solo producto** entre los registros originalmente completos para estas tres columnas. Si la combinación es ambigua, o falta el precio, el producto se deja nulo.

El indicador `Item Fue Recuperado` permite auditar cuáles fueron inferidos.

In [3]:
df_limpio = df.copy()
item_era_nulo = df_limpio["Item"].isna()

conocidos = df.dropna(subset=["Category", "Price Per Unit", "Item"])
n_por_clave = conocidos.groupby(["Category", "Price Per Unit"])["Item"].nunique()
claves_unicas = n_por_clave[n_por_clave.eq(1)].index

# Una correspondencia por clave, usando únicamente las que no son ambiguas.
mapa_productos = (
    conocidos.groupby(["Category", "Price Per Unit"])["Item"]
    .first()
    .loc[claves_unicas]
)

claves_todas = pd.MultiIndex.from_frame(df_limpio[["Category", "Price Per Unit"]])
items_inferidos = pd.Series(claves_todas.map(mapa_productos), index=df_limpio.index)

df_limpio["Item"] = df_limpio["Item"].fillna(items_inferidos)
df_limpio["Item Fue Recuperado"] = item_era_nulo & df_limpio["Item"].notna()

print("Pares categoría/precio unívocos:", len(claves_unicas))
print("Productos originalmente nulos:", int(item_era_nulo.sum()))
print("Productos recuperados:", int(df_limpio["Item Fue Recuperado"].sum()))
print("Productos aún nulos:", int(df_limpio["Item"].isna().sum()))

Pares categoría/precio unívocos: 200
Productos originalmente nulos: 1213
Productos recuperados: 604
Productos aún nulos: 609


## 4. Corregir tipos sin imputar información desconocida

- `Transaction Date` se convierte a fecha y se comprueba que no se introduzcan fechas inválidas.
- `Discount Applied` se convierte a booleano **nullable** (`True` / `False` / `pd.NA`), sin equiparar desconocido con `False`.
- `Quantity` se conserva nula cuando falta y se convierte a entero nullable (`Int64`) únicamente tras comprobar que no hay unidades fraccionarias.
- Se comprueba que los importes no requieren más de dos decimales para el posterior tipo SQL `DECIMAL(18,2)`.

In [4]:
# Fechas
fechas_originales_presentes = df_limpio["Transaction Date"].notna()
df_limpio["Transaction Date"] = pd.to_datetime(
    df_limpio["Transaction Date"], format="%Y-%m-%d", errors="coerce"
)
fechas_invalidas = int((fechas_originales_presentes & df_limpio["Transaction Date"].isna()).sum())
if fechas_invalidas:
    raise ValueError(f"Hay {fechas_invalidas} fechas originales no convertibles. Revisar antes de exportar.")

# Descuentos: tolera booleanos del CSV y sus representaciones textuales.
descuentos_texto = df_limpio["Discount Applied"].astype("string").str.strip().str.lower()
equivalencias = {"true": True, "false": False, "1": True, "0": False}
desconocidos_inesperados = sorted(set(descuentos_texto.dropna()) - set(equivalencias))
if desconocidos_inesperados:
    raise ValueError(f"Estados de descuento no reconocidos: {desconocidos_inesperados}")
df_limpio["Discount Applied"] = descuentos_texto.map(equivalencias).astype("boolean")

# Cantidades: no imputar nulos; verificar enteros antes de cambiar el dtype.
qty = df_limpio["Quantity"].dropna()
if not (qty.mod(1).abs() < 1e-9).all():
    raise ValueError("Se detectaron cantidades fraccionarias: no convertir automáticamente a Int64.")
df_limpio["Quantity"] = df_limpio["Quantity"].astype("Int64")

# No redondear datos sin verificar primero la precisión.
for columna in ["Price Per Unit", "Total Spent"]:
    serie = df_limpio[columna].dropna()
    mas_de_dos_decimales = int((~np.isclose(serie, serie.round(2), rtol=0, atol=1e-9)).sum())
    if mas_de_dos_decimales:
        raise ValueError(f"{columna}: {mas_de_dos_decimales} valores precisan más de dos decimales")

print("Fecha mínima y máxima:", df_limpio["Transaction Date"].min().date(),
      df_limpio["Transaction Date"].max().date())
print("Tipo de Discount Applied:", df_limpio["Discount Applied"].dtype)
display(df_limpio["Discount Applied"].value_counts(dropna=False))

Fecha mínima y máxima: 2022-01-01 2025-01-18
Tipo de Discount Applied: boolean


Discount Applied
True     4219
<NA>     4199
False    4157
Name: count, dtype: Int64

## 5. Auditoría final de calidad

Se contrastan filas, IDs únicos, nulos y consistencia matemática. **`SUM` / `AVG` de importes no equivalen a facturación completa**: quedan transacciones sin `Total Spent`. Se conserva el importe disponible de transacciones sin producto identificado para no perderlo al analizar por categoría.

In [5]:
if df_limpio["Transaction ID"].isna().any() or df_limpio["Transaction ID"].duplicated().any():
    raise AssertionError("Transaction ID debe ser único y no nulo")
if df_limpio.duplicated().any():
    raise AssertionError("Se encontraron filas completamente duplicadas")

completos = df_limpio.dropna(subset=["Price Per Unit", "Quantity", "Total Spent"])
calculado = completos["Price Per Unit"] * completos["Quantity"].astype("float64")
inconsistencias = int((~np.isclose(completos["Total Spent"], calculado, rtol=0, atol=1e-8)).sum())

resumen_final = pd.Series({
    "filas": len(df_limpio),
    "transacciones_unicas": df_limpio["Transaction ID"].nunique(),
    "item_nulos": df_limpio["Item"].isna().sum(),
    "precio_nulos": df_limpio["Price Per Unit"].isna().sum(),
    "cantidad_nulos": df_limpio["Quantity"].isna().sum(),
    "importe_nulos": df_limpio["Total Spent"].isna().sum(),
    "descuento_desconocido": df_limpio["Discount Applied"].isna().sum(),
    "items_recuperados": df_limpio["Item Fue Recuperado"].sum(),
    "registros_comprobados": len(completos),
    "inconsistencias_matematicas": inconsistencias,
    "importe_registrado": df_limpio["Total Spent"].sum(),
    "importe_sin_item_identificado": df_limpio.loc[df_limpio["Item"].isna(), "Total Spent"].sum(),
})
display(resumen_final.to_frame("resultado"))

if inconsistencias:
    raise AssertionError(f"Se encontraron {inconsistencias} inconsistencias matemáticas")

# Conciliación específica del dataset usado durante el proyecto.
esperados = {
    "filas": 12575, "transacciones_unicas": 12575, "item_nulos": 609,
    "precio_nulos": 609, "cantidad_nulos": 604, "importe_nulos": 604,
    "descuento_desconocido": 4199, "items_recuperados": 604,
    "registros_comprobados": 11362, "inconsistencias_matematicas": 0,
    "importe_registrado": 1552071.0, "importe_sin_item_identificado": 79072.5,
}
diferencias = {k: (resumen_final[k], v) for k, v in esperados.items()
               if not np.isclose(float(resumen_final[k]), float(v), rtol=0, atol=1e-7)}
if diferencias:
    raise AssertionError(f"La auditoría difiere de lo comprobado en el proyecto: {diferencias}")
print("Conciliación completada: resultados coinciden con los verificados en el proyecto.")

,resultado
filas,12575.0
transacciones_unicas,12575.0
item_nulos,609.0
precio_nulos,609.0
cantidad_nulos,604.0
importe_nulos,604.0
descuento_desconocido,4199.0
items_recuperados,604.0
registros_comprobados,11362.0
inconsistencias_matematicas,0.0


Conciliación completada: resultados coinciden con los verificados en el proyecto.


## 6. Exportar el CSV limpio para SQL Server

El archivo conserva los nombres de columnas del CSV trabajado. Durante **Import Flat File** en SSMS se renombraron a `Transaction_ID`, `Price_Per_Unit`, etc. Para evitar que el asistente interprete mal los puntos decimales, cargar `Price_Per_Unit` y `Total_Spent` como **`VARCHAR(50)` en `dbo.RetailSales_Prueba`** y convertir después con el script `sql/01_carga_y_validacion.sql`.

**No sobrescribir el CSV original.** Revisar licencia antes de distribuir el archivo limpio.

In [6]:
destino.parent.mkdir(parents=True, exist_ok=True)
df_limpio.to_csv(destino, index=False, encoding="utf-8-sig", date_format="%Y-%m-%d")
print(f"CSV preparado: {destino}")
print("Siguiente etapa: Import Flat File → dbo.RetailSales_Prueba; ejecutar sql/01_carga_y_validacion.sql")

CSV preparado: /mnt/data/retail-sales-analysis/data/processed/retail_store_sales_clean.csv
Siguiente etapa: Import Flat File → dbo.RetailSales_Prueba; ejecutar sql/01_carga_y_validacion.sql
